In [1]:
#!pip install transformers datasets pandas torch accelerate bitsandbytes

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm import tqdm
from huggingface_hub import login

# load in CVS of emails
df = pd.read_csv('../data/raw/extended_emails.csv')

# check if loaded
print(f"Loaded {len(df)} emails")
df.head

#Checking if cuda is avilable
print(torch.cuda.is_available())

login()

# define modle
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it")


# Load the text-generation pipeline
# device="cuda" if you havbe a GPU, otherwise "cpu"

pipe = pipeline(
    "text-generation",
    tokenizer=tokenizer,
    model=model,
    device="cuda",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print(f'Loaded {model}')

Loaded 100 emails
True


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [8]:
def build_reply_prompt(subject, body, category):
    """
    Prompt to draft a professional insurance reply email
    """

    prompt = f"""You are an empathetic & professional customer support agent for a UK Insurance. Your tone is understanding, compasionate and solution-oritated

    Task: Draft a reply that acknowledges the customer's frustrations, asks clarify questions if neccessary and offers immeditate next steps.

    Context: 
    -This is how the email will be presented to you, all the customers issues will be detailed in the Body so this is your context for replying.
    -Subject: {subject}
    -Body: {body}
    -This email is a: {category}

    Format:
    - Be polite and empathetic
    - Use UK English
    - Rasssure that we are here to help
    - Keep it concise (3-4 short paragraphs)
    - Sign off with "Kind regards, Ryan Sylvester, Customer Services"

    Keep the response concise, warm and professional and provide further information for further contact if necessary.
    Reply:"""
    return prompt


In [7]:
def build_classification_prompt(subject, body):
    """
    Few-shot prompt with 3 examples emails before the test email.
    """

    prompt = f"""You are an AI assistant helping an insurance company categorize customer emails.

Categories:
    - claim: customer wants to make and insurance claim
    - complaint: customer is unhappy and wants to raise a complaint
    - renewal: customer has questions about policy renewal
    - policy_change: customer wants to change details on their policy
    - general_query: general questions or requests

Examples:
- Example 1:
    Email Subject: Car accident claim
    Email Body: Hi, I was involved in a minor car accident yesterday. Can you tell me how to start a claim and any other information I need to provide?
    Category: claim

- Example 2:
    Email Subject: Unhappy with claim decision
    Email Body: I recently submitted a claim and received a decision saying the items are not covered. I am very unhappy with this.
    Category: complaint

- Example 3:
    Email Subject: Renewal Query & Auto-renewal opt out
    Email Body: Hi, I think my car insurance is due for renewal soon but I can't find the email. When does my policy end? and How would I turn off auto-renewal so that policy will end on the renewal date?
    Category: renewal

- Example 4:
    Email Subject: No claims discount proof
    Email Body: I am switching to a different insurer and they have asked for proof of my no claims discount. Can you give me details how I would request this?
    Category: general_query

- Example 5:
    Email Subject: Adding a named driver & change of address
    Email Body: Hi, I would like to provide an address update and add a driver to my policy. Coule you let me know what information I would need to provide and how likely my premiums are to go up?
    Category: policy_change

Now classify this email:

Email Subject: {subject}
Email Body: {body}

Taking the above examples into consideration which category out of the list above would you categorize this email with? Pelase only reply with the category of the email (claim, complaint, renewal, policy_change, or general query) and strictly nothing else.

Category:"""

    return prompt


In [ ]:
def classify_email():
    """Run classification and return just the predicited category"""

    # Build & Run all classifications prompts at once
    prompt_classification = [build_classification_prompt(row['email_subject'], row['email_body'])
            for _, row in df.iterrows()]
    classifications = pipe(prompt_classification, max_new_tokens=10, do_sample=False, return_full_text=False, batch_size=64)
    
    # Extract all predictions
    predictions = []
    print(classifications)
    for i, classification in enumerate(classifications):
        try:
            generated = classifications[i][0]['generated_text'].strip()
            prediction = generated.replace('\n', '').replace('*','').strip()
            predictions.append(prediction)

        except (KeyError, IndexError, AttributeError) as e:
            print(f"Error processing email {i}: {e}")
            print(f"Classification output: {classification}")
            predictions.append("unknown")
    # Build & Run all reply prompts at once
    reply_prompts = [build_reply_prompt(df.iloc[i]['email_subject'], df.iloc[i]['email_body'], predictions[i])
                     for i in range(len(df))]
    
    print("Running reply generation batch")
    replies = pipe(reply_prompts, max_new_tokens=200, do_sample=False, return_full_text=False, batch_size=64)

    # Extract all replys
    gen_replies = []
    for i, reply in enumerate(replies):
        print(reply)
        try:
            # reply is also a list containing one dict
            generated_reply = reply[0]['generated_text'].strip()
            gen_replies.append(generated_reply)
        except (KeyError, IndexError, AttributeError) as e:
            print(f"Error processing reply {i}: {e}")
            gen_replies.append("[Error generating reply]")

    # Write results to ./data/processed/prompt_fine_tuning_restuls.txt    
    print ("Writing Results to file...")
    with open('../data/processed/prompt_fine_tuning_results.txt', 'w', encoding='utf-8') as f:

        for i, row in df.iterrows():
            f.write(f"="*80+"\n")
            f.write(f"ID: {row['id']}\n\n")
            f.write(f"Email Subject: {row['email_subject']}\n\n")
            f.write(f"Email Body: {row['email_body']}\n\n")
            f.write(f"True Category: {row['category']}\n")
            f.write(f"Predicted Category: {predictions[i]}\n\n")
            f.write(f"Ideal Reply:\n{row['ideal_reply']}\n\n")
            f.write(f"Generated Reply:\n{gen_replies[i]}\n\n")

    return predictions


df['email_assistant'] = classify_email()

# Calculate accuracy
correct = (df['category'] == df['email_assistant']).sum()
accuracy = correct / len(df)

print(f"Few accuracy: {correct}/{len(df)} = {accuracy:.2%}")
print(f"\nPredictions")
print(df[['id', 'category', 'email_assistant']])

Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[[{'generated_text': ' claim \n'}], [{'generated_text': ' claim \n'}], [{'generated_text': ' claim \n'}], [{'generated_text': ' claim \n'}], [{'generated_text': ' claim\n\n\n'}], [{'generated_text': ' complaint \n\n\n'}], [{'generated_text': ' complaint \n\n\n'}], [{'generated_text': ' complaint \n\n\n'}], [{'generated_text': ' renewal \n'}], [{'generated_text': ' renewal \n'}], [{'generated_text': ' \n**general_query** \n'}], [{'generated_text': ' renewal \n'}], [{'generated_text': ' policy_change \n'}], [{'generated_text': ' policy_change \n'}], [{'generated_text': ' general_query \n'}], [{'generated_text': ' general_query \n'}], [{'generated_text': ' general_query \n'}], [{'generated_text': ' general_query \n'}], [{'generated_text': ' general_query \n'}], [{'generated_text': ' general_query \n'}], [{'generated_text': ' claim \n'}], [{'generated_text': ' claim \n'}], [{'generated_text': ' claim \n'}], [{'generated_text': ' claim \n'}], [{'generated_text': ' complaint \n\n\n'}], [{'ge

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "\n\n    Dear [Customer Name],\n\n    Thank you for contacting us about the water damage in your home. I understand this is a very stressful situation, and I'm so sorry to hear about the flooding.  \n\n    We want to assure you that we're here to help you through this process. To get started, could you please tell me a little more about the incident?  Specifically, what type of damage has occurred, and how long ago did the pipe burst?\n\n    Once I have a clearer picture of the situation, I can guide you through the next steps in the claim process.  In the meantime, please ensure you keep all receipts and documentation related to the repairs and clean-up.\n\n    Kind regards,\n    Ryan Sylvester, Customer Services\n\n\n"}]
Dear [Customer Name],

    Thank you for contacting us about the water damage in your home. I understand this is a very stressful situation, and I'm so sorry to hear about the flooding.  

    We want to assure you that we're here to help you thro